In [1]:
import os
import numpy as np
import pandas as pd
import pywt
from scipy.stats import entropy
from scipy.fft import rfft
from antropy import (
    sample_entropy, spectral_entropy, detrended_fluctuation,
    perm_entropy, katz_fd, higuchi_fd, petrosian_fd, hjorth_params
)
from sklearn.decomposition import FastICA

# ✅ EEG channel names (EMOTIV EPOC+ excluding reference electrodes)
EEG_CHANNELS = ['AF3','F7','F3','FC5','T7','P7','O1',
                'O2','P8','T8','FC6','F4','F8','AF4']

# ✅ Load Valence & Arousal mapping from labels.csv
labels_df = pd.read_csv('labels.csv')
label_map = labels_df.set_index(['Subject', 'Game']).to_dict(orient='index')

def get_labels(subject, game):
    key = (subject, game)
    if key in label_map:
        return label_map[key]['Valence'], label_map[key]['Arousal']
    else:
        print(f'[SKIPPED] No label found for {subject} {game}')
        return None, None

# ✅ Differential entropy
def differential_entropy(signal):
    var = np.var(signal)
    return 0.5 * np.log(2 * np.pi * np.e * var) if var > 0 else 0

# ✅ Build feature CSV header
def make_header(bands):
    basic_features = ['Mean', 'Std', 'Var', 'ZCR', 'Energy', 'Entropy',
                      'LogEntropy', 'SampleEntropy', 'SpecEntropy', 'DFA',
                      'PermEntropy', 'DiffEntropy',
                      'KatzFD', 'HiguchiFD', 'PetrosianFD',
                      'Hjorth_Act', 'Hjorth_Mob', 'Hjorth_Comp']
    cols = [f'{fn}_{b}' for b in bands for fn in basic_features]
    cols += ['TotalWaveletEnergy', 'TotalPSD']
    return 'Subject,Game,Valence,Arousal,' + ','.join(cols) + '\n'

# ✅ Compute features from a list of DWT detail coefficients
def feats_from_details(details):
    features = []
    wavelet_energy = [np.sum(c ** 2) for c in details]
    total_energy = sum(wavelet_energy)
    combined_signal = np.concatenate(details)

    for c in details:
        try:
            f = [
                np.mean(c),
                np.std(c),
                np.var(c),
                np.sum(np.diff(np.sign(c)) != 0),
                np.sum(c**2),
                entropy(np.histogram(c, bins=10)[0] + 1),
                np.sum(np.log(np.square(c) + 1e-12)),
                sample_entropy(c),
                spectral_entropy(c, sf=128, method='fft'),
                detrended_fluctuation(c),
                perm_entropy(c, normalize=True),
                differential_entropy(c),
                katz_fd(c),
                higuchi_fd(c),
                petrosian_fd(c)
            ]
            f += list(hjorth_params(c))
        except:
            f = [0] * 18
        features.extend(f)

    # Add total wavelet energy and total PSD across all levels
    total_psd = np.sum(np.abs(rfft(combined_signal)) ** 2)
    features.append(total_energy)
    features.append(total_psd)
    return features
